# 17. CWT Spectrograms + Dictionary Learning
**Objective:** Validate that Dictionary Learning can parse 2D Time-Frequency spectrograms by flattening the CWT tensors and extracting sparse representations.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

sys.path.append(os.path.abspath('../'))
from src.config import PREPROCESSED_DIR, DEVICE
from src.dictionary_freq import FrequencyDictionaryLearner, FrequencyOMPExtractor
from src.cwt_features import extract_cwt_spectrograms

plt.style.use('seaborn-v0_8-whitegrid')

### 1. Load Data & Generate CWT Features

In [2]:
file_path = os.path.join(PREPROCESSED_DIR, "DB1_subject_1.h5")
with h5py.File(file_path, 'r') as f:
    X_bal = np.array(f['X'])
    y_bal = np.array(f['y']).astype(np.int64)
    reps_bal = np.array(f['reps'])

train_reps = [1, 2, 3, 4, 5, 6, 7]
test_reps = [8, 9, 10]

train_idx = np.where(np.isin(reps_bal, train_reps))[0]
test_idx = np.where(np.isin(reps_bal, test_reps))[0]

# We already verified 16 scales works well
X_cwt = extract_cwt_spectrograms(X_bal, n_scales=16, n_jobs=-1)

# Shape is currently (18630, 16 scales, 20 time, 10 channels)
# We flatten the time and frequency axes to create a 320-dimensional signature per channel
N, S, T, C = X_cwt.shape
X_cwt_flat = X_cwt.reshape(N, S * T, C)

print(f"Flattened CWT Dictionary Input Shape: {X_cwt_flat.shape}")

X_train_cwt, y_train = X_cwt_flat[train_idx], y_bal[train_idx]
X_test_cwt, y_test = X_cwt_flat[test_idx], y_bal[test_idx]

Starting parallel CWT extraction on 18630 windows...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    0.9s
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    0.9s
[Parallel(n_jobs=-1)]: Done  17 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.18981802796690644s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done  37 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done  48 tasks      | elapsed:    1.1s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.08470988273620605s.) Setting batch_size=4.
[Parallel(n_jobs=-1)]: Done  63 tasks      | elapsed:    1.1s
[Parallel(n_jobs=-1)]: Done  88 tasks      | elapsed:    1.2s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.1559600830078125s.) Setting batch_size=8.
[Parallel(n_jobs=-1)]: Done 128 tasks      | elapsed:    1.3s
[Parallel(n_jobs=-1)]: Done 188 tasks      | elapsed:    1.5s
[Parallel(n_jobs

CWT extraction complete. Spectrogram tensor shape: (18630, 16, 20, 10)
Flattened CWT Dictionary Input Shape: (18630, 320, 10)


### 2. Train Mini-Batch Dictionary on CWT Features

In [3]:
# We increase atoms to 64 and non-zero coefficients to 5 to handle the 320D complexity
cwt_learner = FrequencyDictionaryLearner(n_atoms=64, transform_n_nonzero_coefs=5, method='minibatch')
cwt_learner.fit(X_train_cwt)
cwt_learner.save_dictionary("emg_dict_minibatch_cwt_S1.npz")

cwt_extractor = FrequencyOMPExtractor(cwt_learner.dictionary_, n_nonzero_coefs=5)

X_train_sparse = cwt_extractor.transform(X_train_cwt)
X_test_sparse = cwt_extractor.transform(X_test_cwt)

print(f"CWT Sparse Features Shape: {X_train_sparse.shape}")

scaler = StandardScaler()
X_train_sparse_scaled = scaler.fit_transform(X_train_sparse)
X_test_sparse_scaled = scaler.transform(X_test_sparse)

Training MINIBATCH Dictionary with 64 atoms on 126640 frequency spectra...
Dictionary learning complete.
Dictionary saved to /workspaces/TCC/models/emg_dict_minibatch_cwt_S1.npz


/home/vscode/.local/lib/python3.12/site-packages/sklearn/utils/_param_validation.py:191: RuntimeWarning: Orthogonal matching pursuit ended prematurely due to linear dependence in the dictionary. The requested precision might not have been met.
  return func(*args, **kwargs)
/home/vscode/.local/lib/python3.12/site-packages/sklearn/utils/_param_validation.py:191: RuntimeWarning: Orthogonal matching pursuit ended prematurely due to linear dependence in the dictionary. The requested precision might not have been met.
  return func(*args, **kwargs)


CWT Sparse Features Shape: (12664, 640)


### 3. PyTorch Classification on CWT Sparse Codes

In [4]:
class CWTSparseClassifier(nn.Module):
    def __init__(self, input_dim: int, num_classes: int):
        super(CWTSparseClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        return self.net(x)

num_classes = int(max(y_train.max(), y_test.max()) + 1)
input_dim = X_train_sparse_scaled.shape[1]
model = CWTSparseClassifier(input_dim, num_classes).to(DEVICE)

weights = np.ones(num_classes, dtype=np.float32)
classes, counts = np.unique(y_train, return_counts=True)
weights[classes] = 1.0 / counts
tensor_weights = torch.FloatTensor(weights / weights.sum()).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=tensor_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_train_sparse_scaled), torch.LongTensor(y_train)), 
    batch_size=64, shuffle=True
)

print("\nTraining Deep Classifier on CWT Sparse Features...")
for epoch in range(40):
    model.train()
    for bx, by in train_loader:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad()
        out = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    test_x = torch.FloatTensor(X_test_sparse_scaled).to(DEVICE)
    preds = torch.argmax(model(test_x), dim=1).cpu().numpy()
    
from sklearn.metrics import accuracy_score, f1_score
acc_nn = accuracy_score(y_test, preds)
f1_nn = f1_score(y_test, preds, average='macro')

print(f"CWT + Dictionary + PyTorch Accuracy: {acc_nn * 100:.2f}%")
print(f"CWT + Dictionary + PyTorch Macro F1: {f1_nn * 100:.2f}%")


Training Deep Classifier on CWT Sparse Features...
CWT + Dictionary + PyTorch Accuracy: 35.03%
CWT + Dictionary + PyTorch Macro F1: 33.57%
